# 📐 Unidad 1: Óptica y Geometría
## Tema: El Modelo Pinhole y la Distorsión de Lente

¡Hola! 👋 En esta sesión vamos a dejar de ver la imagen como una simple matriz y vamos a entender **cómo llegó esa imagen ahí**.

La presentación nos enseñó el **Modelo Pinhole** (Cámara Estenopeica), que es la simplificación geométrica de cómo una cámara ve el mundo. También nos habló de que los lentes no son perfectos y curvan las líneas rectas (**Distorsión**).

### Objetivo de aprendizaje
Simular matemáticamente cómo un objeto 3D se proyecta en un sensor 2D y cómo un lente puede deformar esa proyección.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuración para gráficos bonitos
%matplotlib inline

### 1. El Modelo Pinhole (De 3D a 2D)

La fórmula mágica que vimos en clase para proyectar un punto del mundo $(X, Y, Z)$ al plano de la imagen $(u, v)$ es:

$$ u = f \cdot \frac{X}{Z} $$
$$ v = f \cdot \frac{Y}{Z} $$

Donde $f$ es la **Distancia Focal**. 

**Experimento:** Vamos a crear un cubo 3D simple y ver qué pasa si cambiamos la distancia focal (el "zoom") o si alejamos el objeto (aumentamos Z).

In [ ]:
def proyeccion_pinhole(puntos_3d, f_focal):
    """
    Proyecta puntos 3D (X, Y, Z) a 2D (u, v) usando el modelo Pinhole.
    """
    puntos_2d = []
    for punto in puntos_3d:
        X, Y, Z = punto
        
        # Evitamos dividir por cero si el objeto está en el ojo de la cámara
        if Z <= 0:
            continue
            
        # APLICAMOS LA FÓRMULA DE PROYECCIÓN (Triángulos semejantes)
        u = f_focal * (X / Z)
        v = f_focal * (Y / Z)
        
        puntos_2d.append([u, v])
    
    return np.array(puntos_2d)

# --- CREAMOS UN CUBO EN 3D ---
# Definimos los 8 vértices de un cubo situado a 10 unidades de distancia (Z=10)
cubo_3d = np.array([
    [-1, -1, 10], [1, -1, 10], [1, 1, 10], [-1, 1, 10], # Cara trasera
    [-1, -1, 12], [1, -1, 12], [1, 1, 12], [-1, 1, 12]  # Cara delantera (más lejos)
])

# Proyectamos con una distancia focal pequeña (Gran Angular, f=200)
proyeccion_f200 = proyeccion_pinhole(cubo_3d, f_focal=200)

# Proyectamos con una distancia focal grande (Teleobjetivo, f=1000)
proyeccion_f1000 = proyeccion_pinhole(cubo_3d, f_focal=1000)

# --- VISUALIZACIÓN ---
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.scatter(proyeccion_f200[:, 0], proyeccion_f200[:, 1], c='red')
plt.title("Focal Corta (f=200)\nEl objeto se ve pequeño")
plt.xlim(-300, 300); plt.ylim(-300, 300); plt.grid(True)
plt.gca().set_aspect('equal')

plt.subplot(1, 2, 2)
plt.scatter(proyeccion_f1000[:, 0], proyeccion_f1000[:, 1], c='blue')
plt.title("Focal Larga (f=1000)\nEfecto Zoom")
plt.xlim(-300, 300); plt.ylim(-300, 300); plt.grid(True)
plt.gca().set_aspect('equal')

plt.show()

### 2. Distorsión Radial (Efecto de Barril)

En la presentación vimos que los lentes reales curvan la luz. Esto se modela matemáticamente con polinomios.

La ecuación simplificada para la distorsión radial es:
$$ r_{distorsionado} = r_{original} (1 + k_1 r^2) $$

Donde $k_1$ es el coeficiente de distorsión. 
* Si $k_1 > 0$: Distorsión de Barril (se infla).
* Si $k_1 < 0$: Distorsión de Cojín (se chupa).

In [ ]:
# Vamos a crear una REJILLA perfecta (como un tablero de ajedrez)
x, y = np.meshgrid(np.linspace(-1, 1, 15), np.linspace(-1, 1, 15))

# Coeficiente de distorsión (Positivo = Barril)
k1 = 0.5 

# Calculamos el radio 'r' (distancia al centro) de cada punto
r = np.sqrt(x**2 + y**2)

# Aplicamos la fórmula de distorsión a las coordenadas
x_dist = x * (1 + k1 * r**2)
y_dist = y * (1 + k1 * r**2)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Mundo Real (Rejilla Perfecta)")
plt.scatter(x, y, s=5, c='black')
plt.grid(True)
plt.gca().set_aspect('equal')

plt.subplot(1, 2, 2)
plt.title(f"Lo que ve la cámara (Distorsión k={k1})")
plt.scatter(x_dist, y_dist, s=5, c='red')
plt.grid(True)
plt.gca().set_aspect('equal')

plt.show()

### Conclusión

Acabas de simular la **Geometría de la Visión**. 
1. Entendiste que la cámara solo hace triángulos semejantes para proyectar el mundo.
2. Viste que las matemáticas pueden explicar por qué las GoPro curvan el horizonte (Distorsión de Barril).